In [1]:
!pip install -q transformers torchaudio librosa soundfile scikit-learn

In [2]:
import torch
import torchaudio
import librosa
import numpy as np

from transformers import AutoModel

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")

PyTorch version: 2.11.0+cpu
CUDA available: False
Using CPU


In [4]:
MODEL_ID = "cmu-mlsp/DELULU"

print("Loading DELULU...")

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = model.to(device)
model.eval()

print("DELULU loaded!")

Loading DELULU...


config.json:   0%|          | 0.00/985 [00:00<?, ?B/s]

configuration_delulu.py:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/cmu-mlsp/DELULU:
- configuration_delulu.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_delulu.py:   0%|          | 0.00/3.28k [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  379MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] DELULUModel LOAD REPORT from: cmu-mlsp/DELULU
Key                               | Status     |  | 
----------------------------------+------------+--+-
logit_generator.final_proj.bias   | UNEXPECTED |  | 
logit_generator.label_embeddings  | UNEXPECTED |  | 
logit_generator.final_proj.weight | UNEXPECTED |  | 
mask_generator.mask_embedding     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DELULU loaded!


In [6]:
print(model)
print('\n\n')
print("Model type:", type(model))
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

DELULUModel(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): FeatureExtractor(
      (conv_layers): ModuleList(
        (0): ConvLayerBlock(
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(4,), bias=False)
        )
        (1-4): 4 x ConvLayerBlock(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        )
        (5-6): 2 x ConvLayerBlock(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        )
      )
    )
    (encoder): Encoder(
      (feature_projection): FeatureProjection(
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (projection): Linear(in_features=512, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (transformer): Transformer(
        (pos_conv_embed): ConvolutionalPositionalEmbedding(
          (conv): ParametrizedConv1d(
            768, 768, kerne

In [11]:
from datasets import load_dataset

dataset = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="train.100",
    streaming=True
)

print("Dataset loaded")

samples = []

speaker_a = []
speaker_b = None

for example in dataset:

    if len(speaker_a) == 0:
        speaker_a.append(example)

    elif example["speaker_id"] == speaker_a[0]["speaker_id"]:
        speaker_a.append(example)

    elif speaker_b is None:
        speaker_b = example

    if len(speaker_a) >= 2 and speaker_b is not None:
        break

print("Speaker A ID:", speaker_a[0]["speaker_id"])
print("Speaker B ID:", speaker_b["speaker_id"])

audio_A1 = speaker_a[0]["audio"]
audio_A2 = speaker_a[1]["audio"]
audio_B1 = speaker_b["audio"]

print("A1:", audio_A1["sampling_rate"], audio_A1["array"].shape)
print("A2:", audio_A2["sampling_rate"], audio_A2["array"].shape)
print("B1:", audio_B1["sampling_rate"], audio_B1["array"].shape)

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Dataset loaded
Speaker A ID: 374
Speaker B ID: 7800
A1: 16000 (232480,)
A2: 16000 (257360,)
B1: 16000 (239600,)


In [12]:
def prepare_audio(audio_dict, target_sr=16000):

    waveform = torch.tensor(
        audio_dict["array"],
        dtype=torch.float32
    )

    # Convert stereo → mono if necessary
    if waveform.ndim > 1:
        waveform = waveform.mean(dim=0)

    original_sr = audio_dict["sampling_rate"]

    # Resample if necessary
    if original_sr != target_sr:
        waveform = torchaudio.functional.resample(
            waveform,
            original_sr,
            target_sr
        )

    # Add batch dimension
    waveform = waveform.unsqueeze(0)

    return waveform

waveform_A1 = prepare_audio(audio_A1)
waveform_A2 = prepare_audio(audio_A2)
waveform_B1 = prepare_audio(audio_B1)

print(waveform_A1.shape)

torch.Size([1, 232480])


In [13]:
def extract_delulu_features(waveform):

    waveform = waveform.to(device)

    with torch.no_grad():
        outputs = model(
            waveform
        )

    return outputs

In [14]:
outputs_A1 = extract_delulu_features(waveform_A1)
features_A1 = outputs_A1.last_hidden_state

print(type(outputs_A1))
print()
print(outputs_A1)
print()

# (batch, no of audio frames, DELULU representation dimension)
print("Shape:", features_A1.shape)


<class 'transformers.modeling_outputs.BaseModelOutput'>

BaseModelOutput(last_hidden_state=tensor([[[ 0.2274,  0.1846, -0.1161,  ...,  0.1877, -0.0233, -0.2548],
         [ 0.2347,  0.1811, -0.1173,  ...,  0.1811, -0.0136, -0.2464],
         [ 0.2714,  0.1914, -0.1797,  ...,  0.1803, -0.0974, -0.2466],
         ...,
         [-0.0459, -0.1156,  0.6152,  ...,  0.1088, -0.3156, -0.3338],
         [ 0.1787,  0.1018,  0.2208,  ...,  0.1950, -0.1830, -0.3612],
         [ 0.2331,  0.1839,  0.0470,  ...,  0.1914, -0.1854, -0.2645]]]), hidden_states=None, attentions=None)

Shape: torch.Size([1, 907, 768])


In [16]:
def delulu_embedding(waveform):

    waveform = waveform.to(device)

    with torch.no_grad():
        outputs = model(waveform)

    features = outputs.last_hidden_state

    # Mean pool across time
    embedding = features.mean(dim=1)

    # L2 normalize
    embedding = torch.nn.functional.normalize(
        embedding,
        p=2,
        dim=1
    )

    return embedding

emb_A1 = delulu_embedding(waveform_A1)
emb_A2 = delulu_embedding(waveform_A2)
emb_B1 = delulu_embedding(waveform_B1)

print("A1:", emb_A1.shape)
print("A2:", emb_A2.shape)
print("B1:", emb_B1.shape)

A1: torch.Size([1, 768])
A2: torch.Size([1, 768])
B1: torch.Size([1, 768])


In [17]:
import torch.nn.functional as F

def cosine_similarity(emb1, emb2):

    return F.cosine_similarity(
        emb1,
        emb2
    ).item()

same_speaker = cosine_similarity(
    emb_A1,
    emb_A2
)

different_speaker = cosine_similarity(
    emb_A1,
    emb_B1
)

print(f"A vs A: {same_speaker:.4f}")
print(f"A vs B: {different_speaker:.4f}")

A vs A: 0.9495
A vs B: 0.3873


In [18]:
def verify_voice(
    enrollment_embedding,
    test_embedding,
    threshold=0.5
):

    score = cosine_similarity(
        enrollment_embedding,
        test_embedding
    )

    verified = score >= threshold

    print(f"Similarity: {score:.4f}")
    print(f"Threshold:  {threshold:.4f}")

    if verified:
        print("✅ VERIFIED")
    else:
        print("❌ REJECTED")

    return score, verified

print("--- Same Speaker ---")

verify_voice(
    emb_A1,
    emb_A2,
    threshold=0.5
)

print("\n--- Different Speaker ---")

verify_voice(
    emb_A1,
    emb_B1,
    threshold=0.5
)

--- Same Speaker ---
Similarity: 0.9495
Threshold:  0.5000
✅ VERIFIED

--- Different Speaker ---
Similarity: 0.3873
Threshold:  0.5000
❌ REJECTED


(0.38731059432029724, False)